# NEMAR nm000109: BIDS recordings and a paired task comparison

Rest and mental-arithmetic EEG in BIDS (Zyma et al. 2019, Data 4(1):14; NEMAR v1.0.0). This notebook does not filter again. The stored headers are not one setting: some files are labelled 0.5–45 Hz and others 0–250 Hz, and a few subjects disagree across the two tasks. Read `metadata['highpass']` and `metadata['lowpass']` on each recording before pooling. The same limits are copied onto each feature row as `highpass_hz` and `lowpass_hz`, with `duration_sec`. Those fields describe the file. They do not mean this notebook applied a filter. The ECG channel is dropped in the cells below. `extract_features` itself keeps it until you drop it. Licence, DOI, and the header counts are in `docs/validation/DATASETS.md`.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from eegmath import extract_features, load
from eegmath.io import iter_recordings
from eegmath.preprocess import drop_channels
from eegmath.signal import adf_stationarity, morlet_power
from eegmath.spectral import welch_psd
from eegmath.stats import compare_groups, correlation, design_summary
from eegmath.viz import plot_psd
from eegmath.workflow import feature_table

def _validation_data() -> Path:
    override = os.environ.get('EEGMATH_DATA')
    if override:
        return Path(override).expanduser()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'external_data').is_dir():
            return candidate / 'external_data'
    return here / 'external_data'

DATA = _validation_data()
ROOT = DATA / 'nemar_nm000109'
def _validation_output() -> Path:
    override = os.environ.get('EEGMATH_VALIDATION_OUT')
    if override:
        return Path(override).expanduser()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'notebooks').is_dir():
            return candidate / 'validation_output'
    return here / 'validation_output'

OUT = _validation_output()
OUT.mkdir(parents=True, exist_ok=True)
assert (ROOT / 'dataset_description.json').exists(), (
    f'NEMAR nm000109 was not found at {ROOT}. See docs/validation/DATASETS.md, '
    'or set EEGMATH_DATA to the directory that contains nemar_nm000109/.'
)
CHANNELS = ['Fz', 'Cz', 'Pz', 'O1', 'O2']

## One subject, both tasks, including a two-file list

In [2]:
pair = load(ROOT, subjects=['00'])
display(pair.describe()[['subject_id', 'condition', 'sfreq', 'n_channels', 'duration_sec', 'units']])
assert {rec.condition for rec in pair} == {'rest', 'mentalArithmetic'}
rest = next(rec for rec in pair if rec.condition == 'rest')
assert rest.ch_types[list(rest.ch_names).index('ECG')] == 'ecg'
assert rest.metadata['highpass'] == 0.5 and rest.metadata['lowpass'] == 45.0
assert int(float(rest.metadata['participant']['age'])) == 21
eeg = drop_channels(rest, ['ECG']).pick_channels(CHANNELS)
assert 'bandpass' not in ' '.join(eeg.provenance)
spectral = extract_features(eeg, ['alpha_relative', 'theta_relative', 'beta_relative', 'rms'])
shown = spectral[['channel', 'alpha_relative', 'theta_relative', 'rms']].copy()
shown['rms_uV'] = shown['rms'] * (1e6 if eeg.units == 'V' else 1.0)
display(shown[['channel', 'alpha_relative', 'theta_relative', 'rms_uV']].round(4))
assert (shown['rms_uV'] > 0.1).all()
relative = spectral[['alpha_relative', 'theta_relative', 'beta_relative']]
assert np.isfinite(relative.to_numpy()).all()
assert ((relative >= 0) & (relative <= 1)).all().all()
freqs, psd = welch_psd(eeg.data, eeg.sfreq)
fig, ax = plot_psd(freqs, psd, ch_names=list(eeg.ch_names), fmax=45)
fig.savefig(OUT / 'nemar_example_psd.png', dpi=120, bbox_inches='tight')
fig.clf()
paths = sorted((ROOT / 'sub-00' / 'eeg').glob('*_eeg.edf'))
listed = feature_table(paths, ['alpha_relative'], )
assert set(listed['condition']) == {'rest', 'mentalArithmetic'}
assert 'ECG' in load(paths[0]).ch_names
cropped = eeg.spawn('crop-10s', data=eeg.data[:, : int(10 * eeg.sfreq)])
power, times = morlet_power(cropped.pick_channels(['Cz']), np.array([6.0, 10.0, 20.0]), n_cycles=5)
assert power.shape[-1] == cropped.n_times
display(pd.Series(adf_stationarity(cropped.data[cropped.ch_names.index('Cz')])))

,subject_id,condition,sfreq,n_channels,duration_sec,units
0,00,mentalArithmetic,500.0,21,62.0,V
1,00,rest,500.0,21,182.0,V


,channel,alpha_relative,theta_relative,rms_uV
0,Fz,0.4074,0.1266,12.0304
1,Cz,0.3082,0.1628,13.5168
2,Pz,0.4401,0.1060,11.8955
3,O1,0.5072,0.0896,12.6902
4,O2,0.6842,0.0542,17.2613


statistic                                                  -8.669927
p_value                                                          0.0
used_lag                                                          32
n_obs                                                           4967
critical_values    {'1%': -3.4316672302646283, '5%': -2.862122072...
null                                                       unit root
note               Rejecting a unit root is not a general demonst...
dtype: object

<Figure size 700x400 with 0 Axes>

## Every subject

Each file is read and reduced to Fz, Cz, Pz, O1, and O2. The subject-level value is the mean of those channel-wise relative powers. It is not the relative power of an averaged spectrum. The paired test below uses one row per subject and task and pools all 36 subjects. That is the comparison recorded in `docs/validation/RESULTS.md`. Subjects whose headers disagree across tasks stay in the pool. The same five-channel mean on the eight subjects labelled 0.5–45 Hz on both tasks is a different test: the difference is −0.081, the Wilcoxon p is 0.078, and the mean-difference interval includes zero. The nineteen subjects labelled 0–250 Hz on both tasks remain significant (difference −0.096, p = 0.0012). `significant` is the corrected rank test. The interval beside it is a paired t interval and can disagree with the flag.

In [3]:
rows = []
for recording in iter_recordings(ROOT):
    usable = drop_channels(recording, ['ECG'], missing='ignore')
    names = [name for name in CHANNELS if name in usable.ch_names]
    table = extract_features(usable.pick_channels(names), ['alpha_relative', 'theta_relative'])
    rows.append(table.groupby(['subject_id', 'condition'], as_index=False)[['alpha_relative', 'theta_relative']].mean())
summary = pd.concat(rows, ignore_index=True)
summary.to_csv(OUT / 'nemar_subject_means.csv', index=False)
display(design_summary(summary, subject='subject_id', condition='condition'))
display(summary.groupby('condition')[['alpha_relative', 'theta_relative']].agg(['mean', 'std', 'count']).round(4))
assert summary['subject_id'].nunique() == 36
assert set(summary['condition']) == {'rest', 'mentalArithmetic'}
result = compare_groups(
    summary, ['alpha_relative', 'theta_relative'], 'condition',
    'mentalArithmetic', 'rest', subject='subject_id', test='wilcoxon',
)
display(result[['feature', 'n_a', 'effect', 'mean_difference_a_minus_b', 'ci_low', 'ci_high', 'p_value', 'p_corrected']])
assert result['n_a'].eq(36).all()
wide = summary.pivot(index='subject_id', columns='condition', values='alpha_relative')
change = wide['mentalArithmetic'] - wide['rest']
ages = pd.read_csv(ROOT / 'participants.tsv', sep='\t', encoding='utf-8-sig')
ages['subject_id'] = ages['participant_id'].str.replace('sub-', '', regex=False)
joined = change.rename('alpha_change').reset_index().merge(ages[['subject_id', 'age']], on='subject_id')
display(pd.Series(correlation(joined['age'].astype(float), joined['alpha_change'], method='spearman')))
result.to_csv(OUT / 'nemar_wilcoxon.csv', index=False)

{'n_rows': 72,
 'n_subjects': 36,
 'rows_per_subject': {'count': 36.0,
  'mean': 2.0,
  'std': 0.0,
  'min': 2.0,
  '25%': 2.0,
  '50%': 2.0,
  '75%': 2.0,
  'max': 2.0},
 'conditions': {'mentalArithmetic': 36, 'rest': 36},
 'max_rows_per_subject_condition': 1,
 'subjects_in_multiple_conditions': 36,
 'complete_subjects': 36}

alpha_relative               theta_relative             
                           mean     std count           mean    std count
condition                                                                
mentalArithmetic         0.2877  0.1366    36         0.1474  0.040    36
rest                     0.3675  0.1375    36         0.1425  0.038    36

,feature,n_a,effect,mean_difference_a_minus_b,ci_low,ci_high,p_value,p_corrected
0,alpha_relative,36,-0.671188,-0.079791,-0.120014,-0.039568,0.000033,0.000067
1,theta_relative,36,0.121706,0.004943,-0.008800,0.018687,0.715245,0.715245


method                      spearman
r                           -0.00203
p_value                     0.990627
n                                 36
ci_low                           NaN
ci_high                          NaN
ci_note    not computed for Spearman
dtype: object

Age in this release is a between-subject covariate of a within-subject alpha change. The correlation is reported with its interval when the method provides one. Spearman is used because the age distribution was not assumed to be linear.